## PyTorch Fundamentals

### PyTorch Tensors 

In [1]:
import torch

In [2]:
X = torch.tensor(
    [
        [1.0, 4.0, 7.0],
        [2.0, 3.0, 6.0]
    ],
)

In [3]:
X

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [4]:
print(X.shape)
print(X.dtype)

torch.Size([2, 3])
torch.float32


In [5]:
print(X[0, 1])
print(X[:,1])

tensor(4.)
tensor([4., 3.])


In [6]:
10 * (X + 1.0)

tensor([[20., 50., 80.],
        [30., 40., 70.]])

In [7]:
print(X.exp())
print(X.mean())
print(X.max(dim=0)) # note: dim 0 means across dim 0 i.e. across rows i.e. max of all the cols

tensor([[   2.7183,   54.5981, 1096.6332],
        [   7.3891,   20.0855,  403.4288]])
tensor(3.8333)
torch.return_types.max(
values=tensor([2., 4., 7.]),
indices=tensor([1, 0, 0]))


In [8]:
X @ X.T

tensor([[66., 56.],
        [56., 49.]])

Converting a tensor into a numpy array.

In [9]:
import numpy as np
X.numpy()

array([[1., 4., 7.],
       [2., 3., 6.]], dtype=float32)

In [10]:
torch.tensor(np.array(
    [
        [1.0, 4.0, 7.0],
        [2.0, 3.0, 6.0]
    ]
))

tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float64)

The default precision for floats is 32 bits in PyTorch, whereas it’s 64 bits in NumPy. It’s generally better to use 32 bits in deep learning. So when calling the torch.tensor() function
to convert a NumPy array to a tensor, it’s best to specify dtype=torch.float32.

In [11]:
torch.tensor(np.array(
    [
        [1.0, 4.0, 7.0],
        [2.0, 3.0, 6.0]
    ]
), dtype=torch.float32)

tensor([[1., 4., 7.],
        [2., 3., 6.]])

You can also modify a tensor in place using indexing and slicing, as with a NumPy array.

In [12]:
X[:, 1] = -99
X

tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]])

In-place operations: abs_(), sqrt_(), zero_(), relu_()  
Regular operations: abs(), sqrt(), zero(), relu()

In [13]:
X.abs_()
X

tensor([[ 1., 99.,  7.],
        [ 2., 99.,  6.]])

### Hardware Acceleration


In [14]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0))

True
1
0
NVIDIA GeForce RTX 4060 Laptop GPU
_CudaDeviceProperties(name='NVIDIA GeForce RTX 4060 Laptop GPU', major=8, minor=9, total_memory=8187MB, multi_processor_count=24, uuid=a682ab30-74b5-8cfa-b6d0-884254cbea06, pci_bus_id=1, pci_device_id=0, pci_domain_id=0, L2_cache_size=32MB)


In [15]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

device

'cuda'

Lets create a tensor on the GPU. One option is to create tensor on the CPU and then copy it to the GPU using the to() method.


In [16]:
M = torch.tensor(
    [
        [1., 2., 3.],
        [4., 5., 6.]
    ]
)
M = M.to(device)

M.device

device(type='cuda', index=0)

Another option is to create tensor directly on the GPU using device argument.


In [17]:
M = torch.tensor(
    [
        [1., 2., 3.],
        [4., 5., 6.]
    ], 
    device=device
)

M.device

device(type='cuda', index=0)

Once the tensor is on the GPU, we can run operations on it normally, and they will all take place on the GPU:

In [18]:
R = M @ M.T
R

tensor([[14., 32.],
        [32., 77.]], device='cuda:0')

Let’s run a little test to compare the speed of a matrix multiplication running on the CPU versus the GPU.

In [19]:
M = torch.rand((1000, 1000))
%timeit M @ M.T

7.99 ms ± 230 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
M = torch.rand((1000, 1000), device=device)
%timeit M @ M.T

308 μs ± 4.33 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


GPU gave around 25 times than the CPU.

## Autograd

In [21]:
x = torch.tensor(5.0 , requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [22]:
f.backward() 
# computes derivative of f wrt every variable it depends on (just x here)
# and stores the derivative in .grad of the respective variables it depends on
x.grad

tensor(10.)

Let's try for x1, x2 and f1, f2.  
Here, when f1.backward(), p.d. of f1 w.r.t x1 calculate, store it in x1.grad, same w.r.t x2.  
When f2.backward(), p.d. of f2 w.r.t x1 calculate, add it to existing value in x1.grad, same w.r.t x2.

In [23]:
x1 = torch.tensor(4.0 , requires_grad=True)
x2 = torch.tensor(5.0 , requires_grad=True)
f1 = x1 ** 2 + x2 ** 3
f2 = x2 ** 2 + x1 ** 3

In [24]:
f1.backward()
f2.backward()
print(x1.grad)
print(x2.grad)

tensor(56.)
tensor(85.)


In [25]:
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad

Whole training loop altogether looks like this:

In [26]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(10):
    f = x ** 2
    f.backward()
    with torch.no_grad():
        x -= learning_rate * x.grad

    x.grad.zero_()

In [27]:
x

tensor(0.5369, requires_grad=True)

## Implementing Linear Regression

### Linear Regression Using Tensors and Autograd

In [61]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
df = data.data
target = data.target

In [62]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df, target, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

Creating tensors for X

In [63]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_valid = torch.tensor(X_valid, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

Scaling the tensors of X manually

In [64]:
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

In [65]:
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_valid = torch.tensor(y_valid, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

Creating paramters of our linear regression model.

In [33]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

Training model using batch gradient descent.

In [34]:
learning_rate = 0.2
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {loss:.2f}")

Epoch 1 / 20, Loss: 16.05
Epoch 2 / 20, Loss: 3.18
Epoch 3 / 20, Loss: 1.59
Epoch 4 / 20, Loss: 1.11
Epoch 5 / 20, Loss: 0.92
Epoch 6 / 20, Loss: 0.83
Epoch 7 / 20, Loss: 0.79
Epoch 8 / 20, Loss: 0.76
Epoch 9 / 20, Loss: 0.74
Epoch 10 / 20, Loss: 0.73
Epoch 11 / 20, Loss: 0.71
Epoch 12 / 20, Loss: 0.70
Epoch 13 / 20, Loss: 0.69
Epoch 14 / 20, Loss: 0.68
Epoch 15 / 20, Loss: 0.67
Epoch 16 / 20, Loss: 0.66
Epoch 17 / 20, Loss: 0.65
Epoch 18 / 20, Loss: 0.65
Epoch 19 / 20, Loss: 0.64
Epoch 20 / 20, Loss: 0.63


In [35]:
with torch.no_grad():
    y_pred_test = X_test @ w + b
((y_pred_test - y_test) ** 2).mean()

tensor(0.6490)

### Linear Regression Using PyTorch's High-Level API

In [36]:
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

In [37]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [38]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [39]:
for param in model.parameters():
    print(param)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


In [40]:
for param in model.named_parameters():
    print(param)

('weight', Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True))
('bias', Parameter containing:
tensor([0.3117], requires_grad=True))


In [41]:
model(X_train[:3])

tensor([[0.0786],
        [0.1613],
        [0.5480]], grad_fn=<AddmmBackward0>)

In [42]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [43]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [44]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.270204544067383
Epoch 2/20, Loss: 1.8919247388839722
Epoch 3/20, Loss: 1.0636117458343506
Epoch 4/20, Loss: 0.7670066952705383
Epoch 5/20, Loss: 0.6579803228378296
Epoch 6/20, Loss: 0.6157494187355042
Epoch 7/20, Loss: 0.5975340604782104
Epoch 8/20, Loss: 0.5881064534187317
Epoch 9/20, Loss: 0.5820196270942688
Epoch 10/20, Loss: 0.577311635017395
Epoch 11/20, Loss: 0.5732660889625549
Epoch 12/20, Loss: 0.5696136355400085
Epoch 13/20, Loss: 0.5662461519241333
Epoch 14/20, Loss: 0.5631145238876343
Epoch 15/20, Loss: 0.5601916909217834
Epoch 16/20, Loss: 0.5574592351913452
Epoch 17/20, Loss: 0.5549024939537048
Epoch 18/20, Loss: 0.5525087118148804
Epoch 19/20, Loss: 0.5502666234970093
Epoch 20/20, Loss: 0.5481656789779663


In [45]:
with torch.no_grad():
    y_pred_test = model(X_test)

((y_pred_test - y_test) ** 2).mean()

tensor(0.5746)

## Implementing a Regression MLP

MLP with 2 hidden layers and 1 output layer.

In [46]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

In [47]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [48]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.971917629241943
Epoch 2/20, Loss: 2.049002170562744
Epoch 3/20, Loss: 1.0055638551712036
Epoch 4/20, Loss: 0.8661571741104126
Epoch 5/20, Loss: 0.7838791608810425
Epoch 6/20, Loss: 0.732201874256134
Epoch 7/20, Loss: 0.6987429261207581
Epoch 8/20, Loss: 0.6764354109764099
Epoch 9/20, Loss: 0.6608064770698547
Epoch 10/20, Loss: 0.6491540670394897
Epoch 11/20, Loss: 0.6398535370826721
Epoch 12/20, Loss: 0.6319313049316406
Epoch 13/20, Loss: 0.6248723864555359
Epoch 14/20, Loss: 0.6183494925498962
Epoch 15/20, Loss: 0.6122236847877502
Epoch 16/20, Loss: 0.6063849329948425
Epoch 17/20, Loss: 0.6007853150367737
Epoch 18/20, Loss: 0.5953795313835144
Epoch 19/20, Loss: 0.5901413559913635
Epoch 20/20, Loss: 0.5850571990013123


## Implementing Mini-Batch Gradient Descent Using DataLoaders

In [49]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=(device == "cuda"))

Using hardware acceleration. For this, we need to move the model to the GPU and at the start of each iteration we must copy each batch to the GPU.

In [50]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 50),
    nn.ReLU(),
    nn.Linear(50, 50),
    nn.ReLU(),
    nn.Linear(50, 1)
)
model = model.to(device)

In [51]:
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()
n_epochs = 20

In [52]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [53]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 0.7697
Epoch 2/20, Loss: 0.4582
Epoch 3/20, Loss: 0.4041
Epoch 4/20, Loss: 0.3836
Epoch 5/20, Loss: 0.3704
Epoch 6/20, Loss: 0.3601
Epoch 7/20, Loss: 0.3531
Epoch 8/20, Loss: 0.3479
Epoch 9/20, Loss: 0.3438
Epoch 10/20, Loss: 0.3381
Epoch 11/20, Loss: 0.3339
Epoch 12/20, Loss: 0.3319
Epoch 13/20, Loss: 0.3284
Epoch 14/20, Loss: 0.3257
Epoch 15/20, Loss: 0.3215
Epoch 16/20, Loss: 0.3188
Epoch 17/20, Loss: 0.3166
Epoch 18/20, Loss: 0.3141
Epoch 19/20, Loss: 0.3109
Epoch 20/20, Loss: 0.3067


In [54]:
X_test = X_test.to(device)
y_test = y_test.to(device)

with torch.no_grad():
    y_test_pred = model(X_test)

((y_test_pred - y_test) ** 2).mean() 

tensor(0.3194, device='cuda:0')

## Model Evaluation

In [57]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))

In [66]:
valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)
valid_mse = evaluate(model, valid_loader, mse)
valid_mse

tensor(0.3302, device='cuda:0')

Using RMSE instead of MSE.

In [70]:
def rmse(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean().sqrt()

evaluate(model, valid_loader, rmse)

tensor(0.5618, device='cuda:0')

This is not accurate as instead of calculating RMSE of the entire validation set, it calculates RMSE of individual batches and then calculates the mean of all these batches.

In [71]:
evaluate(model, valid_loader, mse, 
         aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))

tensor(0.5746, device='cuda:0')

Rather than implement metrics yourself, you may prefer to use the TorchMetrics library (made by the same team as PyTorch Lightning), which provides many well tested streaming metrics. A streaming metric is an object that keeps track of a given metric, and can be updated one batch at a time. 

In [74]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset() # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [75]:
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate_tm(model, valid_loader, rmse)

tensor(0.5736, device='cuda:0')